# Notebook 02 — Exploratory Data Analysis (EDA)

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction using GraphSAGE

### Objective

This notebook performs focused exploratory data analysis on the verified UCI Bank Marketing dataset.

We will investigate:

- Target imbalance
- Numerical feature distributions
- Categorical feature distributions
- Relationships between features and subscription outcome
- Campaign/contact behavior
- Customer financial and demographic patterns
- Previous campaign outcome
- Correlations among numerical variables
- Potentially useful patterns for later feature engineering and graph construction

### Important boundary

This notebook does **not**:

- Train a machine learning model
- Build the heterogeneous graph
- Fit a preprocessing pipeline
- Select a final prediction threshold
- Use the test set for model tuning

The raw dataset remains unchanged.


## Project Input

Expected dataset:

```text
Bank-Marketing-GNN/
└── data/
    └── raw/
        └── bank-full.csv
```

Notebook 01 established that the dataset contains:

- 45,211 rows
- 17 columns
- Target: `y`
- 7 numerical features
- 9 categorical features
- 0 missing cells
- 0 duplicate rows

This notebook independently reloads the raw CSV and verifies the basic schema before EDA.


In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
TARGET_COLUMN = "y"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"

RESULTS_DIR = PROJECT_ROOT / "artifacts" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)


In [ ]:
# ============================================================
# 2. Load and Verify Dataset
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place bank-full.csv in data/raw/."
    )

df = pd.read_csv(DATA_PATH, sep=";")

assert df.shape == (45211, 17), (
    f"Unexpected dataset shape: {df.shape}. "
    "Notebook 01 reported 45,211 rows and 17 columns."
)
assert TARGET_COLUMN in df.columns

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


# 3. Dataset Overview

Start with a compact overview of data types, cardinality, and basic descriptive information.


In [ ]:
# ============================================================
# 3. Overview
# ============================================================

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(dropna=False),
    "missing_values": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

display(overview)

print("\nDataset info:")
df.info()


# 4. Target Distribution

The target `y` indicates whether the customer subscribed to a term deposit.

Because the positive class is expected to be much smaller than the negative class, class imbalance is an important modeling consideration.


In [ ]:
# ============================================================
# 4. Target Distribution
# ============================================================

target_counts = df[TARGET_COLUMN].value_counts()
target_percentages = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

display(target_summary)

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x=TARGET_COLUMN)

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.title("Bank Term Deposit Subscription Distribution")
plt.xlabel("Subscription Outcome")
plt.ylabel("Customer Count")
plt.tight_layout()

target_plot_path = RESULTS_DIR / "eda_target_distribution.png"
plt.savefig(target_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Target plot saved to:", target_plot_path)


In [ ]:
# ============================================================
# 5. Target Imbalance Ratio
# ============================================================

negative_count = int(target_counts.get("no", 0))
positive_count = int(target_counts.get("yes", 0))

if positive_count > 0:
    imbalance_ratio = negative_count / positive_count
else:
    imbalance_ratio = np.inf

print(f"Negative class ('no') : {negative_count:,}")
print(f"Positive class ('yes'): {positive_count:,}")
print(f"Negative/positive ratio: {imbalance_ratio:.2f}:1")


# 5. Numerical Features

The verified dataset contains seven numerical variables.

We inspect:

- Distribution
- Range
- Central tendency
- Potential skewness
- Potential outliers

No transformations are applied.


In [ ]:
# ============================================================
# 6. Numerical Feature Identification
# ============================================================

numerical_features = df.select_dtypes(include=np.number).columns.tolist()

print("Numerical features:")
print(numerical_features)

numerical_summary = df[numerical_features].describe().T
numerical_summary["skewness"] = df[numerical_features].skew()
display(numerical_summary.round(3))


In [ ]:
# ============================================================
# 7. Numerical Distributions
# ============================================================

n_features = len(numerical_features)
n_cols = 3
n_rows = math.ceil(n_features / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 4 * n_rows)
)

axes = np.asarray(axes).reshape(-1)

for ax, column in zip(axes, numerical_features):
    sns.histplot(
        data=df,
        x=column,
        kde=True,
        ax=ax
    )
    ax.set_title(f"Distribution — {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Count")

for ax in axes[n_features:]:
    ax.remove()

plt.tight_layout()

numerical_dist_path = RESULTS_DIR / "eda_numerical_distributions.png"
plt.savefig(numerical_dist_path, dpi=150, bbox_inches="tight")
plt.show()

print("Numerical distribution plot saved to:", numerical_dist_path)


In [ ]:
# ============================================================
# 8. Numerical Boxplots
# ============================================================

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 4 * n_rows)
)

axes = np.asarray(axes).reshape(-1)

for ax, column in zip(axes, numerical_features):
    sns.boxplot(
        data=df,
        x=column,
        ax=ax
    )
    ax.set_title(f"Boxplot — {column}")
    ax.set_xlabel(column)

for ax in axes[n_features:]:
    ax.remove()

plt.tight_layout()

boxplot_path = RESULTS_DIR / "eda_numerical_boxplots.png"
plt.savefig(boxplot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Numerical boxplot saved to:", boxplot_path)


# 6. Categorical Features

Categorical distributions are important for both traditional preprocessing and the planned heterogeneous graph.

The candidate graph entities identified during Notebook 01 are:

```text
job
education
marital
contact
month
housing
loan
default
poutcome
```


In [ ]:
# ============================================================
# 9. Categorical Feature Identification
# ============================================================

categorical_features = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_features = [
    column for column in categorical_features
    if column != TARGET_COLUMN
]

print("Categorical features:")
print(categorical_features)

categorical_cardinality = (
    df[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .to_frame("unique_values")
)

display(categorical_cardinality)


In [ ]:
# ============================================================
# 10. Categorical Distributions
# ============================================================

for column in categorical_features:
    counts = df[column].value_counts(dropna=False)

    plt.figure(figsize=(9, 5))
    ax = sns.countplot(
        data=df,
        y=column,
        order=counts.index
    )

    plt.title(f"Distribution — {column}")
    plt.xlabel("Customer Count")
    plt.ylabel(column)
    plt.tight_layout()

    safe_name = column.replace(" ", "_")
    plot_path = RESULTS_DIR / f"eda_categorical_{safe_name}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"{column}: plot saved to {plot_path}")


# 7. Subscription Rate by Categorical Features

Raw category counts alone can be misleading because categories can have very different sample sizes.

For each categorical feature, calculate:

```text
subscription rate = number of "yes" customers / total customers
```

This helps identify categories associated with higher or lower observed subscription rates.

These are descriptive relationships, **not causal conclusions**.


In [ ]:
# ============================================================
# 11. Subscription Rate by Categorical Feature
# ============================================================

categorical_target_tables = {}

for column in categorical_features:
    table = (
        df.groupby(column, dropna=False)[TARGET_COLUMN]
        .agg(
            customers="count",
            subscriptions=lambda x: (x == "yes").sum(),
            subscription_rate=lambda x: (x == "yes").mean()
        )
        .sort_values("subscription_rate", ascending=False)
    )

    table["subscription_rate_pct"] = (
        table["subscription_rate"] * 100
    ).round(2)

    categorical_target_tables[column] = table

    print(f"\n{'=' * 80}")
    print(f"{column.upper()} — Subscription Rate")
    print(f"{'=' * 80}")
    display(table)


In [ ]:
# ============================================================
# 12. Visualize Subscription Rate by Categorical Feature
# ============================================================

for column, table in categorical_target_tables.items():

    # Avoid clutter for extremely high-cardinality fields.
    plot_table = table.sort_values("subscription_rate_pct")

    plt.figure(figsize=(10, max(5, len(plot_table) * 0.35)))

    sns.barplot(
        x=plot_table["subscription_rate_pct"],
        y=plot_table.index
    )

    plt.title(f"Observed Subscription Rate by {column}")
    plt.xlabel("Subscription Rate (%)")
    plt.ylabel(column)
    plt.tight_layout()

    safe_name = column.replace(" ", "_")
    plot_path = RESULTS_DIR / f"eda_subscription_rate_{safe_name}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"{column}: subscription-rate plot saved to {plot_path}")


# 8. Numerical Features vs Subscription Outcome

Compare the distributions of numerical variables between customers who subscribed and those who did not.

The goal is to identify meaningful distribution differences that may inform later feature engineering.


In [ ]:
# ============================================================
# 13. Numerical Features by Target
# ============================================================

for column in numerical_features:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x=TARGET_COLUMN,
        y=column
    )

    plt.title(f"{column} by Subscription Outcome")
    plt.xlabel("Subscription Outcome")
    plt.ylabel(column)
    plt.tight_layout()

    safe_name = column.replace(" ", "_")
    plot_path = RESULTS_DIR / f"eda_numeric_vs_target_{safe_name}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"{column}: target comparison saved to {plot_path}")


# 9. Campaign and Contact Behavior

Several variables describe the interaction between the bank and the customer.

Particular attention is given to:

- `contact`
- `day`
- `month`
- `duration`
- `campaign`
- `pdays`
- `previous`
- `poutcome`

These variables can be highly predictive, but their availability at prediction time must be considered later to avoid leakage or unrealistic deployment assumptions.


In [ ]:
# ============================================================
# 14. Campaign Variable Summary
# ============================================================

campaign_features = [
    column for column in [
        "contact",
        "day",
        "month",
        "duration",
        "campaign",
        "pdays",
        "previous",
        "poutcome"
    ]
    if column in df.columns
]

print("Campaign/contact features:")
print(campaign_features)


In [ ]:
# ============================================================
# 15. Campaign Feature Subscription Rates
# ============================================================

for column in campaign_features:

    if df[column].dtype == "object":
        table = (
            df.groupby(column)[TARGET_COLUMN]
            .apply(lambda x: (x == "yes").mean() * 100)
            .sort_values(ascending=False)
        )

        display(
            table.to_frame("subscription_rate_pct")
            .round(2)
        )
    else:
        numeric_table = (
            df.groupby(TARGET_COLUMN)[column]
            .agg(["count", "mean", "median", "std", "min", "max"])
            .round(2)
        )

        print(f"\n{column} by target:")
        display(numeric_table)


# 10. Previous Campaign Outcome

`poutcome` describes the outcome of a previous marketing campaign.

This variable is especially relevant for understanding customer history and is also a candidate graph entity.

We inspect both its distribution and observed subscription rate.


In [ ]:
# ============================================================
# 16. Previous Campaign Outcome
# ============================================================

if "poutcome" in df.columns:

    poutcome_summary = (
        df.groupby("poutcome")[TARGET_COLUMN]
        .agg(
            customers="count",
            subscriptions=lambda x: (x == "yes").sum(),
            subscription_rate=lambda x: (x == "yes").mean()
        )
    )

    poutcome_summary["subscription_rate_pct"] = (
        poutcome_summary["subscription_rate"] * 100
    ).round(2)

    display(poutcome_summary.sort_values(
        "subscription_rate_pct",
        ascending=False
    ))


# 11. Financial and Customer Profile Analysis

Inspect the major customer profile variables:

- `balance`
- `housing`
- `loan`
- `default`
- `age`
- `job`
- `education`
- `marital`

The purpose is to understand how financial and demographic groups differ in observed subscription behavior.


In [ ]:
# ============================================================
# 17. Financial / Profile Variables
# ============================================================

profile_features = [
    column for column in [
        "age",
        "balance",
        "default",
        "housing",
        "loan",
        "job",
        "education",
        "marital"
    ]
    if column in df.columns
]

print("Profile features:")
print(profile_features)


In [ ]:
# ============================================================
# 18. Binary Financial Variables
# ============================================================

binary_profile_features = [
    column for column in ["default", "housing", "loan"]
    if column in df.columns
]

for column in binary_profile_features:

    table = (
        df.groupby(column)[TARGET_COLUMN]
        .agg(
            customers="count",
            subscriptions=lambda x: (x == "yes").sum(),
            subscription_rate=lambda x: (x == "yes").mean()
        )
    )

    table["subscription_rate_pct"] = (
        table["subscription_rate"] * 100
    ).round(2)

    print(f"\n{column}")
    display(table)


# 12. Correlation Analysis

Calculate correlations among numerical features.

Correlation is descriptive and does not establish causality.

The target is excluded from the numeric correlation matrix because the raw target is categorical (`yes` / `no`).


In [ ]:
# ============================================================
# 19. Numerical Correlation Matrix
# ============================================================

correlation_matrix = df[numerical_features].corr()

display(correlation_matrix.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix — Numerical Features")
plt.tight_layout()

correlation_plot_path = RESULTS_DIR / "eda_numerical_correlation.png"
plt.savefig(correlation_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Correlation plot saved to:", correlation_plot_path)


# 13. Subscription Rate by Age Group

Age is numerical, but grouped summaries can make patterns easier to interpret.

The bins below are descriptive only and do not become final model features automatically.


In [ ]:
# ============================================================
# 20. Age Groups
# ============================================================

if "age" in df.columns:

    age_bins = [0, 20, 30, 40, 50, 60, 70, 200]
    age_labels = [
        "<=20",
        "21-30",
        "31-40",
        "41-50",
        "51-60",
        "61-70",
        "71+"
    ]

    age_group = pd.cut(
        df["age"],
        bins=age_bins,
        labels=age_labels,
        include_lowest=True
    )

    age_summary = (
        df.assign(age_group=age_group)
        .groupby("age_group", observed=False)[TARGET_COLUMN]
        .agg(
            customers="count",
            subscriptions=lambda x: (x == "yes").sum(),
            subscription_rate=lambda x: (x == "yes").mean()
        )
    )

    age_summary["subscription_rate_pct"] = (
        age_summary["subscription_rate"] * 100
    ).round(2)

    display(age_summary)

    plt.figure(figsize=(9, 5))
    sns.barplot(
        data=age_summary.reset_index(),
        x="age_group",
        y="subscription_rate_pct"
    )

    plt.title("Observed Subscription Rate by Age Group")
    plt.xlabel("Age Group")
    plt.ylabel("Subscription Rate (%)")
    plt.tight_layout()

    age_plot_path = RESULTS_DIR / "eda_subscription_rate_age_group.png"
    plt.savefig(age_plot_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Age-group plot saved to:", age_plot_path)


# 14. Key EDA Findings — Automated Summary

Generate a compact machine-readable summary of the EDA results.

The notebook intentionally reports observations rather than claiming causation.


In [ ]:
# ============================================================
# 21. Automated EDA Summary
# ============================================================

summary = {
    "dataset_shape": {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1])
    },
    "target": {
        "column": TARGET_COLUMN,
        "class_counts": {
            str(k): int(v)
            for k, v in target_counts.items()
        },
        "class_percentages": {
            str(k): float(v)
            for k, v in target_percentages.items()
        },
        "negative_to_positive_ratio": (
            None if not np.isfinite(imbalance_ratio)
            else round(float(imbalance_ratio), 4)
        )
    },
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "candidate_graph_entities": [
        "job",
        "education",
        "marital",
        "contact",
        "month",
        "housing",
        "loan",
        "default",
        "poutcome"
    ],
    "campaign_features_reviewed": campaign_features,
    "profile_features_reviewed": profile_features,
    "data_quality": {
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "constant_columns": [
            column for column in df.columns
            if df[column].nunique(dropna=False) <= 1
        ]
    },
    "leakage_sensitive_variables_for_prediction_time_review": [
        column for column in [
            "duration",
            "campaign",
            "pdays",
            "previous",
            "poutcome"
        ]
        if column in df.columns
    ]
}

summary_path = RESULTS_DIR / "eda_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("EDA summary saved to:", summary_path)
print(json.dumps(summary, indent=2))


# 15. Notebook 02 Final Verification

The notebook is considered successful when:

- [x] Dataset loads successfully.
- [x] Expected shape is confirmed.
- [x] Target distribution is analyzed.
- [x] Numerical distributions are analyzed.
- [x] Categorical distributions are analyzed.
- [x] Subscription rates are calculated for categorical features.
- [x] Numerical features are compared by target.
- [x] Campaign/contact behavior is analyzed.
- [x] Previous campaign outcome is analyzed.
- [x] Financial/profile variables are analyzed.
- [x] Numerical correlations are visualized.
- [x] EDA plots are saved.
- [x] EDA summary JSON is saved.
- [x] No preprocessing or model training has been performed.

## Expected Artifacts

The notebook should create:

```text
artifacts/
└── results/
    ├── eda_target_distribution.png
    ├── eda_numerical_distributions.png
    ├── eda_numerical_boxplots.png
    ├── eda_numerical_correlation.png
    ├── eda_subscription_rate_age_group.png
    ├── eda_summary.json
    └── additional categorical/feature EDA plots
```

Do not proceed to Notebook 03 until the notebook runs successfully and the outputs have been reviewed.


In [ ]:
# ============================================================
# 22. Automated Final Verification
# ============================================================

assert DATA_PATH.exists(), "Raw dataset is missing."
assert df.shape == (45211, 17), f"Unexpected dataset shape: {df.shape}"
assert TARGET_COLUMN in df.columns, "Target column is missing."
assert df.isna().sum().sum() == 0, "Unexpected missing cells detected."
assert df.duplicated().sum() == 0, "Unexpected duplicate rows detected."
assert summary_path.exists(), "EDA summary artifact was not created."

required_plot_paths = [
    RESULTS_DIR / "eda_target_distribution.png",
    RESULTS_DIR / "eda_numerical_distributions.png",
    RESULTS_DIR / "eda_numerical_boxplots.png",
    RESULTS_DIR / "eda_numerical_correlation.png",
]

missing_plots = [
    str(path)
    for path in required_plot_paths
    if not path.exists()
]

assert not missing_plots, (
    "Required EDA plots are missing: " + ", ".join(missing_plots)
)

print("=" * 75)
print("NOTEBOOK 02 VERIFICATION PASSED")
print("=" * 75)
print(f"Dataset shape      : {df.shape}")
print(f"Target             : {TARGET_COLUMN}")
print(f"Numerical features : {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Missing cells      : {int(df.isna().sum().sum())}")
print(f"Duplicate rows     : {int(df.duplicated().sum())}")
print(f"EDA summary        : {summary_path}")
print(f"Results directory   : {RESULTS_DIR}")
print("=" * 75)
print("Notebook 02 complete. Review the EDA before proceeding to NEXT.")
